In [4]:
import chromadb
from chromadb.utils import embedding_functions

In [5]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

In [6]:
client = chromadb.Client()

collection_name = "my_grocery_collection"

In [7]:
try:
    # Place your database operations inside this block
    collection = client.create_collection(
        name=collection_name,
        metadata={"description":"A collection for storing grocery data"},
        configuration={
            "hnsw" : {"space" : "cosine"},
            "embedding_function":ef
        }
    )
    print("Collection Created")
    pass
except Exception as error:  # Catch any errors and log them to the console
    print(f"Error: {error}")


Collection Created


In [8]:
# Array of grocery-related text items
texts = [
    'fresh red apples',
    'organic bananas',
    'ripe mangoes',
    'whole wheat bread',
    'farm-fresh eggs',
    'natural yogurt',
    'frozen vegetables',
    'grass-fed beef',
    'free-range chicken',
    'fresh salmon fillet',
    'aromatic coffee beans',
    'pure honey',
    'golden apple',
    'red fruit'
]


In [9]:
ids = [f"food_{index+1}" for index, _ in enumerate(texts)]
ids

['food_1',
 'food_2',
 'food_3',
 'food_4',
 'food_5',
 'food_6',
 'food_7',
 'food_8',
 'food_9',
 'food_10',
 'food_11',
 'food_12',
 'food_13',
 'food_14']

In [10]:
collection.add(
    documents=texts,
    metadatas=[
        {"source":"grocery_store", "category" :"food"} for _ in texts
    ],
    ids=ids
)

In [11]:
collection

Collection(name=my_grocery_collection)

In [12]:
all_items = collection.get()
all_items

{'ids': ['food_1',
  'food_2',
  'food_3',
  'food_4',
  'food_5',
  'food_6',
  'food_7',
  'food_8',
  'food_9',
  'food_10',
  'food_11',
  'food_12',
  'food_13',
  'food_14'],
 'embeddings': None,
 'documents': ['fresh red apples',
  'organic bananas',
  'ripe mangoes',
  'whole wheat bread',
  'farm-fresh eggs',
  'natural yogurt',
  'frozen vegetables',
  'grass-fed beef',
  'free-range chicken',
  'fresh salmon fillet',
  'aromatic coffee beans',
  'pure honey',
  'golden apple',
  'red fruit'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'category': 'food', 'source': 'grocery_store'},
  {'source': 'grocery_store', 'category': 'food'},
  {'source': 'grocery_store', 'category': 'food'},
  {'source': 'grocery_store', 'category': 'food'},
  {'category': 'food', 'source': 'grocery_store'},
  {'category': 'food', 'source': 'grocery_store'},
  {'category': 'food', 'source': 'grocery_store'},
  {'source': 'grocery_store', 'category': 'food'},

In [15]:
# Function to perform a similarity search in the collection
def perform_similarity_search(collection, all_items):
    try:
        # Place your similarity search code inside this block
        query_term = "apple"
        results = collection.query(
            query_texts=[query_term],
            n_results=3
        )
        if not results or not results['ids'] or len(results['ids'][0]) == 0:
            print(f'No documents found similar to "{query_term}"')
            return
        else:
            print(f'Top 3 similar documents to "{query_term}":')
            # Access the nested arrays in 'results["ids"]' and 'results["distances"]'
            for i in range(min(3, len(results['ids'][0]))):
                doc_id = results['ids'][0][i]  # Get ID from 'ids' array
                score = results['distances'][0][i]  # Get score from 'distances' array
                # Retrieve text data from the results
                text = results['documents'][0][i]
                if not text:
                 print(f' - ID: {doc_id}, Text: "Text not available", Score: {score:.4f}')
                else:
                    print(f' - ID: {doc_id}, Text: "{text}", Score: {score:.4f}')

        pass
    except Exception as error:
        print(f"Error in similarity search: {error}")

perform_similarity_search(collection,all_items)

Top 3 similar documents to "apple":
 - ID: food_13, Text: "golden apple", Score: 0.3825
 - ID: food_1, Text: "fresh red apples", Score: 0.4809
 - ID: food_14, Text: "red fruit", Score: 0.5965
